# NL-to-SQL Pipeline — Dissertation Walkthrough

This notebook traces a single natural language question through every stage of the pipeline described in **Chapter 3** of the dissertation. Each section explains the *why* behind each design decision, not just the *what*.

---

**Research context.**  
The dissertation investigates whether **few-shot in-context learning (k=3)** or **QLoRA fine-tuning** better suits open-source 7–8B parameter LLMs for SQL generation, under the constraint of a single Google Colab GPU (~24 GB VRAM). Two models are compared:

- **Meta-Llama-3-8B-Instruct** (Grattafiori et al., 2024) [25]
- **Qwen2.5-7B-Instruct** (Qwen et al., 2025) [17]

Evaluation uses the **Spider benchmark** [22] — a human-labelled, cross-domain text-to-SQL dataset — against the **classicmodels** MySQL database.

---

**What this notebook demonstrates** (no model or database connection required — model output is hardcoded):

| Section | Module | What it answers |
|---------|--------|-----------------|
| 1 — Schema representation | `core/schema.py` | How do you describe a database schema in a prompt? |
| 2 — Few-shot prompt construction | `core/prompting.py` | How does in-context learning work mechanically? |
| 3 — SQL extraction | `core/llm.py` | How do you extract valid SQL from noisy LLM output? |
| 4 — Post-processing | `core/postprocess.py` | Why strip `ORDER BY` before scoring with EX? |
| 5 — Pre-execution validation | `core/validation.py` | What errors are caught before the query hits the DB? |
| 6 — Validation scope | `core/validation.py` | What validation deliberately does *not* check. |
| 7 — Execution-guided repair | `agent/react_pipeline.py` | How does the ReAct loop correct itself? |
| 8 — Metrics and results | `evaluation/eval.py` | What do VA / EM / EX / TS each actually measure? |
| 9 — Statistical tests | `scripts/generate_research_comparison.py` | Why Wilcoxon? Why paired? Why BH-FDR? |
| 10 — Summary | — | The dissertation claim and what the numbers say. |

In [ ]:
import os, sys, shutil
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_dir = Path("/content/NLtoSQL")
    if not (repo_dir / "nl2sql").exists():
        if repo_dir.exists():
            shutil.rmtree(repo_dir)
        os.system(f"git clone https://github.com/MacKenzieOBrian/NLtoSQL.git {repo_dir}")
    os.chdir(repo_dir)
    os.system("pip -q install -r requirements.txt")

sys.path.insert(0, str(Path.cwd()))
print("working directory:", Path.cwd())
print("in Colab:", IN_COLAB)

In [ ]:
# real imports — no torch, no DB connection needed for these
from nl2sql.core.prompting   import make_few_shot_messages
from nl2sql.core.postprocess import normalize_sql, first_select_only, guarded_postprocess, RANKING_HINT_RE
from nl2sql.core.validation  import parse_schema_text, validate_sql
from nl2sql.core.llm         import extract_first_select
from nl2sql.agent.agent_tools import schema_to_text

print("imports ok")

---
## 1. Schema Representation

**The problem:** an LLM has no knowledge that your database exists. You must describe the entire schema in the prompt — every table name, every column name — within a token budget.

**Why *not* use full DDL?**  
`CREATE TABLE customers (customerNumber INT NOT NULL AUTO_INCREMENT PRIMARY KEY, ...)` wastes tokens on data types, constraints, and `NULL`/`NOT NULL` annotations that a 7–8B model rarely uses correctly. The compact format encodes only what matters: table names and column names.

```
customers(customerNumber, customerName, country, creditLimit)
orders(orderNumber, customerNumber, status, orderDate)
Join hints: orders.customerNumber = customers.customerNumber
```

**Why prioritise PK and name-like columns first?**  
`build_schema_summary()` ([`core/schema.py`](../nl2sql/core/schema.py)) reorders each table's columns so that primary-key columns and columns whose names match `/(name|id|code|key|type|no\b)/i` appear first. The motivation is *schema-linking* — the observation (Lin et al. [2], Wang et al. [18]) that surfacing join-relevant identifiers early in the token stream helps the model correctly resolve multi-table queries. Transformer attention is position-sensitive; `customerNumber` appearing as the first column of `orders` makes a cross-table join more likely to be generated correctly.

**Join hints** are appended as a separate line from foreign key metadata. Without them, small models frequently emit unconstrained cross-joins (`FROM customers, orders`) instead of explicit `JOIN … ON` expressions.

The cell below constructs schema text from a hardcoded dict that mirrors the structure returned by `build_schema_summary()` at runtime.

In [ ]:
# fake schema dict — same structure as what build_schema_summary() returns
# (nl2sql/core/schema.py queries INFORMATION_SCHEMA at runtime)
schema_cache = {
    "tables": [
        {"name": "customers",    "columns": [{"name": "customerNumber"}, {"name": "customerName"}, {"name": "country"}, {"name": "creditLimit"}]},
        {"name": "orders",       "columns": [{"name": "orderNumber"}, {"name": "customerNumber"}, {"name": "status"}, {"name": "orderDate"}]},
        {"name": "orderdetails", "columns": [{"name": "orderNumber"}, {"name": "productCode"}, {"name": "quantityOrdered"}, {"name": "priceEach"}]},
        {"name": "products",     "columns": [{"name": "productCode"}, {"name": "productName"}, {"name": "productLine"}, {"name": "MSRP"}]},
        {"name": "payments",     "columns": [{"name": "customerNumber"}, {"name": "checkNumber"}, {"name": "amount"}, {"name": "paymentDate"}]},
    ],
    "foreign_keys": [
        {"table": "orders",    "column": "customerNumber", "ref_table": "customers", "ref_column": "customerNumber"},
        {"table": "payments",  "column": "customerNumber", "ref_table": "customers", "ref_column": "customerNumber"},
        {"table": "orderdetails", "column": "orderNumber", "ref_table": "orders",    "ref_column": "orderNumber"},
    ],
}

schema_text = schema_to_text(schema_cache)
print("schema_to_text() ->")
print()
print(schema_text)

---
## 2. Few-Shot In-Context Learning (ICL)

**What ICL is.**  
Large language models trained on internet-scale text have implicitly learned input→output mappings for thousands of task formats. You can *activate* that knowledge without any gradient updates by demonstrating the format in the prompt itself — this is in-context learning (Brown et al. [8]).

For SQL generation, the prompt structure becomes:

```
[schema]

Question: How many customers are in France?
SQL: SELECT COUNT(*) FROM customers WHERE country = 'France';

Question: What is the total payment amount received?
SQL: SELECT SUM(amount) FROM payments;

Question: Top 3 countries by number of orders.   ← new question
SQL:                                               ← model completes this
```

The model sees the `Question → SQL` pattern *k* times, then completes the final turn. No weights change. **This is the entire mechanism.**

**Why k=3 specifically?**  
Gao et al. [23] benchmark k=1,2,3,5 on models of comparable size and find diminishing returns beyond k=3, with prompt length growing linearly. k=0 (the baseline) provides no style template, so EM is near-zero: the model writes valid SQL but in unpredictable style that rarely matches the gold string character-for-character.

**The k=0→k=3 jump on EM is dramatic.**  
At k=0, EM is 0.5% (Llama) and 1.0% (Qwen). At k=3 it rises to 29.8% and 35.7%. This is not because the model becomes 60× more likely to write *correct* SQL — it's because the exemplars teach it the expected *style* (alias conventions, `COUNT(*)` vs `COUNT(col)`, `JOIN` vs subquery). EM measures textual identity; EX measures semantic correctness.

**Seed control and reproducibility.**  
Each condition runs with three seeds (7, 17, 27). For each seed, exemplars are sampled deterministically — `random.Random(seed)` in the baseline, `random.Random(f"{seed}:{nlq}")` in the ReAct loop. Fixing the seed makes results reproducible and gives n=600 (200 questions × 3 seeds) paired observations per condition for the Wilcoxon test.

In [ ]:
NLQ = "Top 3 countries by number of orders."

EXEMPLARS = [
    {"nlq": "List all customer names in Spain.",
     "sql": "SELECT customerName FROM customers WHERE country = 'Spain';"},
    {"nlq": "How many products are in each product line?",
     "sql": "SELECT productLine, COUNT(*) FROM products GROUP BY productLine;"},
    {"nlq": "What is the total payment amount received?",
     "sql": "SELECT SUM(amount) FROM payments;"},
]

msgs_k0 = make_few_shot_messages(schema=schema_text, exemplars=[],        nlq=NLQ)
msgs_k3 = make_few_shot_messages(schema=schema_text, exemplars=EXEMPLARS, nlq=NLQ)

def show_messages(msgs, label):
    print(f"--- {label} ({len(msgs)} messages) ---")
    for m in msgs:
        body = m['content'][:100].replace('\n', ' | ')
        print(f"  [{m['role'].upper():<10}]  {body}")
    print()

show_messages(msgs_k0, "k=0")
show_messages(msgs_k3, "k=3")

print(f"k=0: {len(msgs_k0)} messages  /  k=3: {len(msgs_k3)} messages")
print(f"each exemplar adds 2 messages (user + assistant), so k=3 adds 6")

---
## 3. SQL Extraction from Raw Model Output

**The problem: models are verbose.**  
Instruction-tuned models are trained to be helpful — they explain their reasoning, add caveats, and often wrap code in markdown fences. For evaluation, you need the raw SQL string and nothing else.

Typical raw output from a 7–8B model:

```
Sure! Here is the SQL for your question:

```sql
SELECT c.country, COUNT(o.orderNumber) AS order_count
FROM customers c
JOIN orders o ON c.customerNumber = o.customerNumber
GROUP BY c.country ORDER BY order_count DESC LIMIT 3;
```

This joins the customers and orders tables and returns the top 3 countries.
```

`extract_first_select()` ([`core/llm.py`](../nl2sql/core/llm.py)) handles this:

1. **Strip markdown fences** — removes ` ```sql ` and ` ``` ` wrappers
2. **Scan for SELECT** — uses `SQL_START_RE` to find all positions where `SELECT` appears
3. **Validate the FROM clause** — for each candidate, extracts the table name after `FROM` using `_read_from_target()`. If the table name is a prose stopword (`the`, `a`, `this`, `my`, `our`…) it rejects the candidate as a hallucination
4. **Return the first valid hit** — or `None` if no valid SQL is found (caller falls back to the raw string)

**Why the stopword check?**  
Models sometimes generate pseudo-SQL like `SELECT the answer FROM the database` — syntactically it looks like a SELECT statement but it references no real table. The stopword list catches this without needing a grammar parser.

**In `generate_sql_from_messages()`**, a `_StopOnSemicolon` stopping criterion also halts generation at the first `;` token, so the model can't generate a second statement or explanatory prose after the SQL ends. This is enforced at the token level — the generation loop terminates before the model can continue.

In [ ]:
# typical raw output from a small model — explanation prefix + trailing prose
RAW_NOISY = """Sure! Here is the SQL for your question:

SELECT c.country, COUNT(o.orderNumber) AS order_count
FROM customers c
JOIN orders o ON c.customerNumber = o.customerNumber
GROUP BY c.country ORDER BY order_count DESC LIMIT 3;

This joins the customers and orders tables and returns the top 3 countries."""

print("raw model output:")
for line in RAW_NOISY.strip().split('\n'):
    print(f"  {line}")
print()

extracted = extract_first_select(RAW_NOISY)
print(f"extract_first_select() -> {extracted}")
print()

# prose-only text — no valid SELECT — returns None
RAW_NO_SQL = "I don't know how to answer that question."
print(f"no-SQL text     -> {extract_first_select(RAW_NO_SQL)!r}  (None = fallback to raw text)")

# markdown fences are stripped before scanning
RAW_FENCED = "```sql\nSELECT productName FROM products;\n```"
print(f"fenced SQL      -> {extract_first_select(RAW_FENCED)}")

# prose-FROM is rejected ("from the results" is not a table)
RAW_PROSE = "I can answer from the results I have."
print(f"prose FROM      -> {extract_first_select(RAW_PROSE)!r}  (rejected — 'the' is a stopword)")

---
## 4. Post-Processing: Why Strip `ORDER BY`?

**The EX metric and result-set comparison.**  
Execution accuracy (EX) compares the result sets of the predicted and gold queries as multisets:

```python
Counter(pred_rows) == Counter(gold_rows)
```

A `Counter` has no order — it only counts occurrences. So `[(France, 10), (USA, 8)]` and `[(USA, 8), (France, 10)]` are identical under this comparison. This means **row order does not affect EX**.

**The problem with spurious `ORDER BY`.**  
Most Spider questions do not specify ordering. But small models — especially at k=0 — frequently append `ORDER BY col DESC` to their output. This doesn't *hurt* EX directly (the Counter still matches), but it causes two failure modes:

1. If the gold query has no `ORDER BY`, the gold DB execution returns rows in natural (arbitrary) order, which may differ from the predicted ordering. The Counter comparison still passes — but the semantic intent was to return an unordered set.
2. Some SQL engines apply different query plans depending on `ORDER BY` presence, changing which rows are returned at all for certain query structures.

`_strip_order_by_limit()` ([`core/postprocess.py`](../nl2sql/core/postprocess.py)) removes `ORDER BY` and `LIMIT` unless `RANKING_HINT_RE` fires — a regex checking for words like `top`, `highest`, `lowest`, `most`, `ranked`, `first N`. If the question asks "top 3 countries", the ordering is semantically required and is preserved.

**Why this is conservative.**  
Stripping can occasionally hurt a borderline query that needs ordering for correctness. But the alternative — leaving all `ORDER BY` clauses in — creates more false EX failures from tie-breaking inconsistencies. The conservative choice is to strip unless explicitly asked to rank.

> **Note:** Primary baseline/QLoRA runs use `eval_profile = model_only_raw` which applies `guarded_postprocess()`. This cell demonstrates the utilities directly.

In [ ]:
# NLQ with no ranking keyword → strip_order_by_limit should fire (ORDER BY removed)
NLQ_PP = "What is the total amount paid by each customer?"
SQL_PP  = "SELECT customerNumber, SUM(amount) AS total, paymentDate FROM payments GROUP BY customerNumber ORDER BY total DESC;"

print("=== no ranking keyword in NLQ ===")
print(f"input:  {SQL_PP}")
print(f"NLQ:    {NLQ_PP}")
print()

step1 = first_select_only(SQL_PP)
print(f"step 1 — first_select_only:  {'CHANGED' if step1 != SQL_PP else 'no change'}")

ranking_match = RANKING_HINT_RE.search(NLQ_PP)
print(f"step 2 — RANKING_HINT_RE:    {'fires — keep ORDER BY' if ranking_match else 'no match — strip ORDER BY / LIMIT'}")

result = guarded_postprocess(SQL_PP, NLQ_PP)
print(f"\nguarded_postprocess() -> {result}")
print()

# contrast: NLQ that does mention a ranking cue → ORDER BY preserved
NLQ_RANK = "What are the top 3 countries by total payments?"
result_rank = guarded_postprocess(SQL_PP, NLQ_RANK)
ranking_match2 = RANKING_HINT_RE.search(NLQ_RANK)
print(f"=== ranking keyword present ===")
print(f"NLQ:    {NLQ_RANK}")
print(f"RANKING_HINT_RE fires: {bool(ranking_match2)}  (keyword: '{ranking_match2.group(0)}')")
print(f"guarded_postprocess() -> {result_rank}")
print("ORDER BY and LIMIT preserved — ranking NLQ detected")

---
## 5. Pre-Execution Validation

**Two-stage correctness checking.**  
The pipeline separates *static validation* (before the DB) from *execution* (at the DB). This is intentional: catching obvious errors statically is cheaper and enables the ReAct repair loop to act on structured error reasons rather than opaque DB error messages.

`validate_sql()` ([`core/validation.py`](../nl2sql/core/validation.py)) runs five sequential guard checks:

| Check | What it catches | Reject reason |
|-------|----------------|---------------|
| Empty string | model returned `""` | `empty` |
| `clean_candidate_with_reason()` | no valid SELECT, forbidden DML tokens | `no_select`, `forbidden_sql` |
| `SELECT *` guard | star-projection without "all columns" in NLQ | `select_star_forbidden` |
| `parse_schema_text()` fails | schema text is empty or unparseable | `schema_missing` |
| `schema_validate()` | `FROM` / `JOIN` references a table not in the schema | `unknown_table:X` |

**The table-name check.**  
`schema_validate()` uses regex to extract every table reference in the SQL (`FROM t`, `JOIN t`, `FROM (subquery) AS t`). It checks each against the set of known table names from `parse_schema_text()`. This catches hallucinated table names — one of the most common failure modes for small models at k=0.

**What it does NOT check** (by design):
- Column existence within tables — would require a full column index and would over-reject valid aliased expressions
- SQL syntax errors — caught at execution time
- Semantic correctness — only EX/TS can judge this

The output is `{"valid": bool, "reason": str}` — structured so the ReAct loop can dispatch a targeted repair hint (e.g. the `select_star_forbidden` reason triggers a specific repair message).

In [ ]:
# first see what parse_schema_text does with the schema string
tables, table_cols = parse_schema_text(schema_text)
print("parse_schema_text() ->")
print(f"  tables: {sorted(tables)}")
print()
for tbl, cols in sorted(table_cols.items()):
    print(f"  {tbl}: {sorted(cols)}")
print()

# test three SQL statements against the schema index
test_cases = [
    ("good",
     "SELECT c.country, COUNT(*) FROM customers c JOIN orders o ON c.customerNumber = o.customerNumber GROUP BY c.country ORDER BY COUNT(*) DESC LIMIT 3;"),
    ("bad — 'country' not in orders",
     "SELECT country, COUNT(*) FROM orders GROUP BY country ORDER BY COUNT(*) DESC LIMIT 3;"),
    ("bad — unknown table 'invoices'",
     "SELECT customerNumber FROM invoices WHERE amount > 1000;"),
]

print("validate_sql() results:")
for label, sql in test_cases:
    r = validate_sql(sql, schema_text, nlq=NLQ)
    status = "PASS" if r['valid'] else f"FAIL  [{r['reason']}]"
    print(f"\n  {label}")
    print(f"  {status}")
    print(f"  sql: {sql}")

---
## 6. Validation Scope — What It Deliberately Ignores

`validate_sql()` is intentionally narrow. This is not a limitation — it is a design choice.

**Column-level validation is omitted.** `parse_schema_text()` returns both the table set and a column index (as `_`), but `schema_validate()` is currently called with `{}` as the column index. Checking whether `c.country` exists on `customers` is correct in isolation, but:

- Aliases (`SELECT c.country FROM customers c`) require alias resolution
- Subqueries create virtual columns  
- Expressions like `COUNT(*) AS n` introduce computed columns

A column-level validator that over-rejects valid aliased expressions would hurt EX more than the hallucinated-column errors it catches. The conservative choice is to leave column validation to execution.

**SQL syntax checking is omitted.** MySQL's parser is the ground truth. Re-implementing syntax rules in Python would be a maintenance burden and would lag behind MySQL's actual behaviour. If the SQL is syntactically invalid, `QueryRunner.run()` catches the `sqlalchemy` exception and records `success=False` — which feeds into the ReAct repair loop.

**Semantic correctness is out of scope.** A query that passes validation and executes successfully may still return the wrong answer. Only EX (compare result sets) and TS (test-suite accuracy across 3 DB variants) can assess semantic correctness.

In [ ]:
# validate_sql focuses on schema-level validity
sql_pass = "SELECT c.country, COUNT(*) FROM customers c JOIN orders o ON c.customerNumber = o.customerNumber GROUP BY c.country ORDER BY COUNT(*) DESC LIMIT 3;"
sql_fail = "SELECT o.country, COUNT(*) FROM orders o GROUP BY o.country ORDER BY COUNT(*) DESC LIMIT 3;"  # 'country' is not a column on orders

for label, sql in [("passing", sql_pass), ("failing (country not in orders)", sql_fail)]:
    r = validate_sql(sql, schema_text, nlq=NLQ)
    status = "PASS" if r['valid'] else f"FAIL  [{r['reason']}]"
    print(f"  {label}")
    print(f"  {status}")
    print(f"  {sql}")
    print()

In [ ]:
# another schema-level check: unknown table reference
sql_good = "SELECT customerName FROM customers WHERE country = 'USA';"
sql_bad  = "SELECT customerName FROM customerz WHERE country = 'USA';"

for label, sql in [("known table", sql_good), ("unknown table", sql_bad)]:
    r = validate_sql(sql, schema_text)
    status = "PASS" if r['valid'] else f"FAIL  [{r['reason']}]"
    print(f"  {label}")
    print(f"  {status}")
    print(f"  {sql}")
    print()

---
## 7. Execution-Guided Repair — The ReAct Loop

**The ReAct framework** (Yao et al. [19]) interleaves *reasoning* and *acting*: after every action the model observes the result and uses it to decide what to do next. For SQL generation this means:

```
Thought:  "I need to find the top 3 countries by order count."
Action:   generate_sql → SELECT country FROM orders GROUP BY country ...
Observe:  validate_sql → FAIL: unknown_table (country is in customers, not orders)
Thought:  "I referenced the wrong table. I need to JOIN."
Action:   repair_sql  → SELECT c.country, COUNT(*) FROM customers c JOIN orders o ...
Observe:  validate_sql → PASS
Action:   run_sql     → {success: True, rowcount: 3}
Observe:  success — stop.
```

**Why `max_repairs=2`?**  
Budget 1 covers a *validation repair* (schema error caught before execution). Budget 2 covers an *execution-guided repair* (runtime SQL error caught after hitting the DB). If `max_repairs=1`, any validation error consumes the entire budget and you never get an execution-guided repair — which is the whole point of the ReAct loop.

**Why zero-shot repair?**  
The generation exemplars are `(NLQ, correct_SQL)` pairs. The repair task requires `(NLQ, bad_SQL, error_message) → corrected_SQL` — a completely different format. Showing generation exemplars during repair confuses the model about its task. DIN-SQL [5] validates this: zero-shot repair with explicit error context outperforms mismatched few-shot repair. The repair prompt is therefore always zero-shot, regardless of the `few_shot_k` setting.

**The trace.** Every step is recorded in `state.trace` (a list of dicts). This makes every pipeline decision auditable — you can inspect which NLQs failed, how many repairs were used, and at which step the loop stopped. The JSON result files in `results/agent/runs/` store the full trace for all 200 test items.

The cell below simulates the repair scenario with hardcoded responses.

In [ ]:
# attempt 1 — first SQL the model produces
attempt_1 = "SELECT country, COUNT(*) FROM orders GROUP BY country ORDER BY COUNT(*) DESC LIMIT 3;"
print("attempt 1:")
print(f"  {attempt_1}")
r1 = validate_sql(attempt_1, schema_text, nlq=NLQ)
print(f"  validate_sql -> {r1}")
print()

# walkthrough-only: manual hint text to illustrate what the repair prompt can include
hint = f"[{r1['reason']}] -- schema shows 'country' is in 'customers', not in 'orders'"
print("illustrative hint in repair prompt:")
print(f"  {hint}")
print()

# attempt 2 — "model" produces corrected SQL (hardcoded here)
attempt_2 = "SELECT c.country, COUNT(*) AS n FROM customers c JOIN orders o ON c.customerNumber = o.customerNumber GROUP BY c.country ORDER BY n DESC LIMIT 3;"
print("attempt 2 (repaired):")
print(f"  {attempt_2}")
r2 = validate_sql(attempt_2, schema_text, nlq=NLQ)
print(f"  validate_sql -> {r2}")
print()
print("repairs used: 1 of 2  (max_repairs=2 in ReactAblationConfig default)")

In [ ]:
# EM scoring — normalize_sql() strips semicolons, lowercases, collapses whitespace
gold_sql = "SELECT c.country, COUNT(*) FROM customers c JOIN orders o ON c.customerNumber = o.customerNumber GROUP BY c.country ORDER BY COUNT(*) DESC LIMIT 3;"

print("normalize_sql() output:")
print(f"  gold:      {normalize_sql(gold_sql)}")
print(f"  attempt_1: {normalize_sql(attempt_1)}")
print(f"  attempt_2: {normalize_sql(attempt_2)}")
print()

print("EM / EX per attempt:")
for label, pred, ex in [
    ("attempt_1 (schema error)", attempt_1, False),
    ("attempt_2 (repaired)",     attempt_2, True),
]:
    em = normalize_sql(pred) == normalize_sql(gold_sql)
    print(f"  {label}:  EM={em}  EX={ex}  VA=True")
print()
print("attempt_2 has EX=True (returns same rows) but EM=False (different alias 'n' vs no alias)")
print("-> EX is the primary metric; EM is too strict for equivalent SQL")

In [ ]:
from collections import Counter

# --- live demonstration of the EX (execution accuracy) comparison ---
# In real evaluation, these rows come from executing SQL against the DB.
# Here we hardcode representative result sets to show the Counter logic.

gold_rows    = [("France", 10), ("USA", 8), ("Germany", 6)]
pred_exact   = [("France", 10), ("USA", 8), ("Germany", 6)]   # same order
pred_reorder = [("USA", 8), ("Germany", 6), ("France", 10)]   # different order → still EX=1
pred_wrong   = [("France", 10), ("USA", 8), ("Spain", 5)]     # wrong country → EX=0

print("EX metric: Counter(pred_rows) == Counter(gold_rows)")
print()
for label, pred in [("exact match",     pred_exact),
                    ("different order",  pred_reorder),
                    ("wrong answer",     pred_wrong)]:
    ex = Counter(pred) == Counter(gold_rows)
    print(f"  {label:<20}  EX={ex}   pred={pred}")

print()
print("EX is order-insensitive: reordered rows still pass.")
print("EX catches wrong VALUES regardless of ordering.")
print()
print("EM comparison (normalize_sql strips semicolons, lowercases, collapses whitespace):")
gold_sql = "SELECT c.country, COUNT(*) FROM customers c JOIN orders o ON c.customerNumber = o.customerNumber GROUP BY c.country ORDER BY COUNT(*) DESC LIMIT 3;"
equiv_sql = "select c.country, count(*)  from  customers c join orders o on c.customernumber = o.customernumber group by c.country order by count(*) desc limit 3;"
print(f"  EM(gold, gold):   {normalize_sql(gold_sql) == normalize_sql(gold_sql)}")
print(f"  EM(gold, equiv):  {normalize_sql(gold_sql) == normalize_sql(equiv_sql)}  (same after normalization)")
diff_alias = "SELECT c.country, COUNT(*) AS n FROM customers c JOIN orders o ON c.customerNumber = o.customerNumber GROUP BY c.country ORDER BY n DESC LIMIT 3;"
print(f"  EM(gold, alias):  {normalize_sql(gold_sql) == normalize_sql(diff_alias)}  (different alias 'n' → EM=False despite same result)")

---
## 8. Evaluation Metrics and Results

**Four metrics, each measuring something different:**

| Metric | What it checks | Implementation |
|--------|---------------|---------------|
| **VA** | Did the SQL execute without a DB error? | `QueryRunner.run().success` |
| **EM** | Is `normalize_sql(pred) == normalize_sql(gold)`? | Textual identity after normalisation |
| **EX** | Does `Counter(pred_rows) == Counter(gold_rows)`? | Result-set equality, order-insensitive |
| **TS** | Does EX hold across 3 DB variants with different data? | Test-suite evaluation [21] |

**EM vs EX — why both matter.**  
At k=0, EX is ~41–49% while EM is near zero. The model writes semantically correct SQL but in unpredictable style. At k=3, both rise — but EM rises more than EX because exemplars teach style as well as correctness. The gap between EM and EX is always positive (EX≥EM), since any exact match is also an execution match, but the converse isn't true.

**TS is the strictest metric.**  
A query that accidentally returns the correct answer for one particular dataset (the primary DB) but is logically wrong will likely fail on at least one of the 3 variant DBs. TS is therefore more reliable than EX for confirming semantic correctness, at the cost of requiring multiple DB variants. Only k=3 conditions are TS-eligible (k=0 VA is too low for TS to be informative).

Mean scores below are computed across **3 seeds × 200 items = 600 scored pairs** per condition.

In [ ]:
# all 8 conditions — means from results/analysis/stats_mean_median_std.csv
conditions = [
    ("Llama Base  k=0", 0.765, 0.005, 0.410, None),
    ("Llama Base  k=3", 0.870, 0.298, 0.517, 0.559),
    ("Llama QLoRA k=0", 0.815, 0.000, 0.440, None),
    ("Llama QLoRA k=3", 0.855, 0.285, 0.465, None),
    ("Qwen  Base  k=0", 0.805, 0.010, 0.480, None),
    ("Qwen  Base  k=3", 0.892, 0.357, 0.610, 0.667),
    ("Qwen  QLoRA k=0", 0.805, 0.010, 0.435, None),
    ("Qwen  QLoRA k=3", 0.895, 0.302, 0.530, 0.566),
]

print(f"  {'Condition':<20}  {'VA':>6}  {'EM':>6}  {'EX':>6}  {'TS':>6}")
print(f"  {'-'*20}  {'------':>6}  {'------':>6}  {'------':>6}  {'------':>6}")
for label, va, em, ex, ts in conditions:
    ts_str = f"{ts:.3f}" if ts else "  N/A"
    flag   = "  <--" if "k=3" in label else ""
    print(f"  {label:<20}  {va:>6.3f}  {em:>6.3f}  {ex:>6.3f}  {ts_str:>6}{flag}")

---
## 9. Statistical Analysis — Test Choices and Interpretation

**Why Wilcoxon signed-rank, not a t-test?**  
EX and EM are binary (0 or 1 per item) — Bernoulli-distributed, not Normal. Shapiro-Wilk formally rejects normality for every condition (expected). The **Wilcoxon signed-rank test** is non-parametric: it ranks absolute differences between paired items and tests whether positive differences outweigh negatives. It makes no normality assumption (Dror et al. [24]; SciPy [29]).

The **paired t-test** is run as corroboration — the CLT makes the t-statistic asymptotically valid at n=600. When Wilcoxon and t-test agree on all 12 EX comparisons (they do), the conclusion is robust.

**Why paired?**  
Each test item has one prediction from condition A and one from condition B, for the *same NLQ*. Pairing cancels question-difficulty variance, giving substantially more statistical power than an unpaired test.

**Why Benjamini-Hochberg FDR, not Bonferroni?**  
Bonferroni divides α=0.05 by 12 comparisons → α=0.0042, assuming all tests are independent. But the same condition (e.g. Llama Base k=3) appears in multiple comparisons, so they are correlated. BH-FDR controls the *false discovery rate* (expected fraction of significant results that are wrong) rather than family-wise error. Less conservative, standard for correlated comparisons in NLP [24].

**Cohen's d_z = mean(A−B) / std(A−B).**  
Computed on the paired differences. d=0.2 small / 0.5 medium / 0.8 large. Distinguishes *statistically significant but tiny* effects from *practically meaningful* differences.

The table below shows the key planned comparisons (all p-values BH-adjusted).

In [ ]:
# paired t-test summary — from results/analysis/stats_paired_ttests.csv
tests = [
    ("Llama Base k0->k3",     "EX", 0.410, 0.517, 1.10e-11, True),
    ("Llama QLoRA k0->k3",    "EX", 0.440, 0.465, 0.0547,   False),   # NOT significant
    ("Qwen Base k0->k3",      "EX", 0.490, 0.610, 8.21e-09, True),
    ("Qwen QLoRA k0->k3",     "EX", 0.435, 0.530, 1.37e-09, True),
    ("Llama Base vs QLoRA k3","EX", 0.517, 0.465, 7.40e-04, True),    # core claim
    ("Qwen Base vs QLoRA k3", "EX", 0.610, 0.530, 8.60e-06, True),    # core claim
]

print(f"  {'comparison':<26}  {'M':<3}  {'left':>6}  {'right':>6}  {'delta':>7}  {'p':>10}  sig?")
print(f"  {'-'*26}  ---  {'------':>6}  {'------':>6}  {'-------':>7}  {'----------':>10}  ----")
for comp, m, lv, rv, p, sig in tests:
    delta = (rv - lv) * 100
    p_str = f"{p:.2e}" if p < 0.001 else f"{p:.4f}"
    note  = "yes" if sig else "NO (p=0.055)"
    print(f"  {comp:<26}  {m:<3}  {lv:>6.3f}  {rv:>6.3f}  {delta:>+6.1f}pp  {p_str:>10}  {note}")

print()
print("Llama QLoRA k0->k3 EX gain is NOT significant — fine-tuning may suppress ICL benefit")
print()

# ReAct (3 seeds: 7, 17, 27)
print("ReAct mean vs Llama Base k=3  (3 seeds, n=600 matched items):")
for m, bv, rv in zip(["VA","EM","EX","TS"], [0.870,0.298,0.517,0.559], [0.925,0.458,0.628,0.661]):
    print(f"  {m}: {bv:.3f} -> {rv:.3f}  ({(rv-bv)*100:+.1f}pp)")
print("  [ReAct means computed from seeds 7, 17, 27]")

---
## 10. Summary — The Dissertation Claim

**The core finding:**  
> **Few-shot in-context learning (k=3) outperforms QLoRA fine-tuning at k=3 on both models, statistically significantly.**

| Comparison | EX delta | p (BH) | Significant? |
|-----------|---------|--------|-------------|
| Llama Base k=3 vs Llama QLoRA k=3 | +5.2pp | 0.0007 | Yes |
| Qwen Base k=3 vs Qwen QLoRA k=3 | +8.0pp | <0.001 | Yes |

This holds under all three metrics (EX, EM, TS) and both Wilcoxon and t-test.

**Why does fine-tuning hurt?**  
QLoRA constrains adaptation to a low-rank subspace (Hu et al. [11]). This specialises the model toward a fixed SQL-generation pattern, competing with the ICL signal carried by the exemplars (Biderman et al. [10]). The Llama result is the clearest evidence: Llama QLoRA k0→k3 EX uplift (+2.5pp) *fails to reach significance* (p=0.055), whereas the base model's equivalent uplift (+10.7pp) is highly significant (p<0.001). Fine-tuning didn't just underperform — it suppressed the model's ability to benefit from in-context examples.

**The k=0→k=3 jump is the strongest signal.**  
EM at k=0: 0.5% (Llama), 1.0% (Qwen). EM at k=3: 29.8% (Llama), 35.7% (Qwen). This +29–35pp jump from adding three examples — with no gradient updates — is the headline result for the ICL chapter.

**ReAct as a third strategy.**  
The execution-guided ReAct loop adds a further +11.2pp EX over Llama Base k=3 and +4.8pp over Qwen Base k=3. ReAct recovers errors that static validation and generation alone cannot: runtime SQL errors caught by the live DB feed directly into the repair prompt. This is the execution-feedback loop from DIN-SQL [5] and ExCoT [6] made explicit.

**Limitations to disclose in the dissertation:**
- Colab VRAM ceiling forces QLoRA over full fine-tuning — results may not generalise to full fine-tuning
- Llama QLoRA k=3 has no TS data (`ts_enabled=False` in that run)
- Exemplar RNG differs between baseline and ReAct — limits direct statistical comparability of those two conditions
- PICARD constrained decoding [15] tested but disabled (requires grammar server incompatible with Colab)